# Explore SeisLM representations

Compare pretrained and randomly initialized SeisLM encoders on **64 real ETHZ earthquake/noise windows**, then visualize pooled features with t-SNE.

[Open in Colab](https://colab.research.google.com/drive/150HS1pyoY4FLnVRksc9E9Bw4UI2q7NtQ)

These plots illustrate representations, not detection accuracy. ETHZ was included in pretraining; these are development windows.

## 1. Install

In fresh Colab **Python 3** (CPU or GPU), run from the top. Python 3.11–3.13 is supported. Restart if requested.

In local Jupyter, run `uv sync --locked --extra notebooks --group notebook`, select that environment, and skip this installation cell.

In [ ]:
import importlib
import importlib.metadata as package_metadata
import subprocess
import sys

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("SeisLM 0.2.1 supports Python 3.11–3.13.")
_loaded_versions = {}
for _module, _package in {
    "numpy": "numpy", "torch": "torch", "seisLM": "seisLM",
    "matplotlib": "matplotlib", "pandas": "pandas", "sklearn": "scikit-learn",
    "h5py": "h5py", "scipy": "scipy", "requests": "requests", "plotly": "plotly",
}.items():
    if _module in sys.modules:
        try:
            _loaded_versions[_package] = package_metadata.version(_package)
        except package_metadata.PackageNotFoundError:
            pass
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet",
     "https://huggingface.co/ltlltl2006/seislm-base/resolve/13a3dcd6ae56c39f5b0750b817c8788a7fc3c99b/seislm-0.2.1-py3-none-any.whl",
     "matplotlib==3.11.2", "pandas==2.3.3", "scikit-learn==1.9.1", "h5py==3.16.0", "requests==2.34.2", "plotly==7.1.0"],
    check=True,
)
importlib.invalidate_caches()
_changed = [name for name, old in _loaded_versions.items() if package_metadata.version(name) != old]
if _changed:
    raise RuntimeError("Updated loaded libraries: " + ", ".join(_changed) + ". Restart the kernel once, then run from the top.")

## 2. Choose model and layer

Start with **base**. `auto` selects an available GPU. `HIDDEN_LAYER=-1` selects the final output; `0` selects the transformer input. Intermediate outputs are also available. Every layer's pooled features are saved.

In [ ]:
import os
from pathlib import Path

MODEL_NAME = os.environ.get("SEISLM_NOTEBOOK_MODEL", "base")  # base / large
DEVICE = os.environ.get("SEISLM_NOTEBOOK_DEVICE", "auto")  # auto / cpu / cuda
SAMPLES_PER_CLASS = int(os.environ.get("SEISLM_NOTEBOOK_SAMPLES_PER_CLASS", "32"))
HIDDEN_LAYER = int(os.environ.get("SEISLM_NOTEBOOK_HIDDEN_LAYER", "-1"))
BATCH_SIZE = 4
SEED = 42
OUTPUT_DIR = Path(os.environ.get("SEISLM_NOTEBOOK_OUTPUT_DIR", "seislm_feature_outputs")).expanduser().resolve()
DATA_DIR = Path(os.environ.get("SEISLM_NOTEBOOK_DATA_DIR", "~/.cache/seislm/ethz-demo")).expanduser().resolve()
CACHE_DIR = os.environ.get("SEISLM_NOTEBOOK_CACHE_DIR") or None
RELEASES = {
    "base": {"repo_id": "ltlltl2006/seislm-base", "revision": "a9e9f79af940a3ea28a49ef9fa495ae89b48d576", "hidden_size": 240},
    "large": {"repo_id": "ltlltl2006/seislm-large", "revision": "3ad2a723faeded0ca45b555c9c0ea8823affcc4f", "hidden_size": 768},
}
if MODEL_NAME not in RELEASES or DEVICE not in {"auto", "cpu", "cuda"}:
    raise ValueError("Choose model base/large and device auto/cpu/cuda.")
if not 8 <= SAMPLES_PER_CLASS <= 64:
    raise ValueError("Choose 8–64 windows per class for this small demonstration.")
release = RELEASES[MODEL_NAME]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import collections
import hashlib
import importlib.metadata
import io
import json
import platform
import time
import uuid

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import torch
from filelock import FileLock
from scipy.signal import resample_poly
from sklearn.manifold import TSNE
from threadpoolctl import threadpool_limits
from seisLM import load_pretrained
from seisLM.model.foundation.multidim_wav2vec2 import MultiDimWav2Vec2ForPreTraining

%matplotlib inline

started = time.perf_counter()
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA was requested but is unavailable. Select a GPU runtime or choose DEVICE='cpu'.")
device = "cuda" if DEVICE == "auto" and torch.cuda.is_available() else DEVICE
if device == "auto":
    device = "cpu"
versions = {"python": platform.python_version(), **{
    name: importlib.metadata.version(name)
    for name in ("seisLM", "torch", "numpy", "pandas", "scipy", "scikit-learn", "h5py", "matplotlib", "plotly")
}}
print("Device:", device, torch.cuda.get_device_name() if device == "cuda" else platform.machine())
print("Model:", release["repo_id"])
print("SeisLM:", versions["seisLM"], "| Python:", versions["python"])
print("Outputs:", OUTPUT_DIR)

## 3. Load ETHZ windows

Earthquake and noise windows are paired from the same recording. Select complete, spike-free development records at 100 or 200 Hz; filter and downsample 200 Hz data to **100 Hz**. Each window has **3,000 samples in Z, N, E order**, with no padding or missing-component filling.

The first run downloads about 17 MB of benchmark tables and only selected waveform portions, avoiding the full 23 GB archive. Subsequent runs reuse a numeric cache.

Sources: [ETHZ](https://seisbench.readthedocs.io/en/main/pages/data/benchmark_datasets.html#ethz) and [task-1 window labels](https://github.com/seisbench/pick-benchmark).

In [ ]:
# Download fixed benchmark tables and read selected HDF5 byte ranges.
TABLES = {
    "metadata.csv": ("datasets/ethz/metadata.csv", 12926120, "542d2e9cfc35847668521fe335bc955b00a42f59f06eaa921ffb92f6009a1dee"),
    "task1.csv": ("auxiliary/pick-benchmark/targets/ethz/task1.csv", 4272225, "7d89c574424aeea09c3e7d609b00386e94ff07a2fa0cd6df7bd1a5fbf98a528c"),
}
MIRROR = "https://seisbench.gfz.de/mirror/"
WAVEFORM_URL = MIRROR + "datasets/ethz/waveforms.hdf5"
WAVEFORM_BYTES = 23248089008

def sha256_file(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

def get_table(name):
    relative, expected_size, expected_sha = TABLES[name]
    destination = DATA_DIR / name
    if destination.exists():
        if destination.stat().st_size == expected_size and sha256_file(destination) == expected_sha:
            return destination
        raise RuntimeError(f"Cached {name} failed its checksum. Remove {destination} and rerun.")
    temporary = destination.with_name(f".{destination.name}.{uuid.uuid4().hex}.download")
    try:
        size = 0
        with requests.get(MIRROR + relative, stream=True, timeout=(15, 60)) as response:
            response.raise_for_status()
            with temporary.open("wb") as handle:
                for chunk in response.iter_content(1024 * 1024):
                    size += len(chunk)
                    if size > expected_size:
                        raise RuntimeError(f"Unexpected size for {name}.")
                    handle.write(chunk)
        if size != expected_size or sha256_file(temporary) != expected_sha:
            raise RuntimeError(f"The released {name} checksum changed; no data were used.")
        temporary.replace(destination)
    finally:
        temporary.unlink(missing_ok=True)
    return destination

class BoundedHTTPFile(io.RawIOBase):
    """A seekable reader that refuses full-archive or unbounded transfers."""
    def __init__(self, url, size):
        super().__init__()
        self.url, self.size, self.position = url, size, 0
        self.bytes_received = self.request_count = 0
        self.session = requests.Session()
        self.cache = {}
        self.max_bytes = 32 * 1024 * 1024
        self.max_requests = 2048

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.position

    def seek(self, offset, whence=0):
        position = offset if whence == 0 else self.position + offset if whence == 1 else self.size + offset if whence == 2 else -1
        if not 0 <= position <= self.size:
            raise ValueError("Invalid remote file position.")
        self.position = position
        return position

    def read(self, size=-1):
        if size < 0 or size > 2 * 1024 * 1024:
            raise RuntimeError("Refusing an unbounded or oversized waveform read.")
        size = min(size, self.size - self.position)
        if size == 0:
            return b""
        key = (self.position, size)
        if key not in self.cache:
            if self.bytes_received + size > self.max_bytes or self.request_count >= self.max_requests:
                raise RuntimeError("Small-demo download budget exceeded. Reduce SAMPLES_PER_CLASS.")
            end = self.position + size - 1
            headers = {"Range": f"bytes={self.position}-{end}", "Accept-Encoding": "identity"}
            with self.session.get(self.url, headers=headers, stream=True, timeout=(15, 60)) as response:
                expected_range = f"bytes {self.position}-{end}/{self.size}"
                if response.status_code != 206 or response.headers.get("Content-Range") != expected_range:
                    raise RuntimeError("Waveform host did not honor the byte range; refusing a full download.")
                if response.headers.get("Content-Encoding", "identity") != "identity":
                    raise RuntimeError("Unexpected compressed HTTP response.")
                value = response.raw.read(size + 1)
                if len(value) != size:
                    raise RuntimeError("Truncated or oversized waveform response.")
            self.bytes_received += len(value)
            self.request_count += 1
            self.cache[key] = value
        value = self.cache[key]
        self.position += len(value)
        return value

    def readinto(self, target):
        value = self.read(len(target))
        target[:len(value)] = value
        return len(value)

    def close(self):
        self.session.close()
        super().close()

In [ ]:
cache_key = f"ethz-task1-dev-v1-n{SAMPLES_PER_CLASS}-seed{SEED}"
cache_npz = DATA_DIR / f"{cache_key}.npz"
cache_json = DATA_DIR / f"{cache_key}.json"
cache_settings = {"version": 1, "samples_per_class": SAMPLES_PER_CLASS, "seed": SEED, "split": "dev", "samples": 3000}
with FileLock(str(DATA_DIR / f"{cache_key}.lock"), timeout=300):
    cache_hit = cache_npz.exists() and cache_json.exists()
    if cache_hit:
        data_manifest = json.loads(cache_json.read_text(encoding="utf-8"))
        if data_manifest["settings"] != cache_settings or sha256_file(cache_npz) != data_manifest["npz_sha256"]:
            raise RuntimeError(f"Invalid cached sample. Remove {cache_npz} and {cache_json}, then rerun.")
        with np.load(cache_npz, allow_pickle=False) as saved:
            raw_windows = saved["waveforms"].copy()
            labels = saved["labels"].copy()
    else:
        metadata = pd.read_csv(get_table("metadata.csv"), low_memory=False)
        targets = pd.read_csv(get_table("task1.csv"))
        complete = metadata[
            (metadata["split"] == "dev") & metadata["trace_sampling_rate_hz"].isin([100.0, 200.0])
            & (metadata["trace_completeness"] >= 0.999) & ~metadata["trace_has_spikes"].astype(bool)
        ].set_index("trace_name", verify_integrity=True)
        targets = targets[(targets["trace_split"] == "dev") & targets["trace_name"].isin(complete.index)]
        paired = targets.groupby("trace_name")["trace_type"].agg(list)
        candidates = sorted(name for name, types in paired.items() if sorted(types) == ["earthquake", "noise"])
        if len(candidates) < SAMPLES_PER_CLASS:
            raise RuntimeError("Not enough complete paired ETHZ records.")
        chosen = np.random.default_rng(SEED).choice(candidates, size=SAMPLES_PER_CLASS, replace=False).tolist()
        windows, label_values, records = [], [], []
        with BoundedHTTPFile(WAVEFORM_URL, WAVEFORM_BYTES) as remote:
            # HDF5 must close before its underlying HTTP file object.
            with h5py.File(remote, "r") as archive:
                component_order = archive["data_format/component_order"][()].decode()
                dimension_order = archive["data_format/dimension_order"][()].decode()
                if component_order != "ZNE" or dimension_order != "CW":
                    raise RuntimeError("The ETHZ archive's component or dimension order changed.")
                for name in chosen:
                    row = complete.loc[name]
                    rate = int(row["trace_sampling_rate_hz"])
                    factor = rate // 100
                    bucket, location = name.split("$", 1)
                    pieces = location.split(",")
                    record_index, components, samples = int(pieces[0]), int(pieces[1][1:]), int(pieces[2][1:])
                    if components != 3:
                        raise ValueError("Expected all three waveform components.")
                    selected = targets[targets["trace_name"] == name].set_index("trace_type")
                    for label, kind in enumerate(("earthquake", "noise")):
                        target = selected.loc[kind]
                        start, end = int(target["start_sample"]), int(target["end_sample"])
                        if int(target["sampling_rate"]) != 100 or end - start != 3000:
                            raise ValueError("Unexpected task-1 window sampling or length.")
                        native_start, native_end = start * factor, end * factor
                        if not 0 <= native_start < native_end <= samples:
                            raise ValueError("Selected window would require padding; choose complete data.")
                        # Include anti-alias filter context before cutting the 100 Hz window.
                        halo = 20 if factor == 2 else 0
                        read_start = max(0, native_start - halo)
                        read_end = min(samples, native_end + halo)
                        native = np.asarray(archive[f"data/{bucket}"][record_index, :3, read_start:read_end])
                        if factor == 2:
                            converted = resample_poly(native, up=1, down=2, axis=-1)
                        else:
                            converted = native
                        offset = (native_start - read_start) // factor
                        window = converted[:, offset:offset + 3000].astype(np.float32)
                        if window.shape != (3, 3000) or not np.isfinite(window).all() or np.any(window.std(axis=-1) == 0):
                            raise ValueError("Selected waveform is incomplete, non-finite, or has an empty component.")
                        windows.append(window)
                        label_values.append(label)
                        records.append({
                            "trace_name": name, "trace_name_original": str(row["trace_name_original"]),
                            "source_id": str(row["source_id"]), "trace_type": kind,
                            "start_sample_100hz": start, "end_sample_100hz": end,
                            "original_sampling_rate_hz": rate,
                        })
            transfer_bytes, transfer_requests = remote.bytes_received, remote.request_count
        raw_windows = np.stack(windows)
        labels = np.asarray(label_values, dtype=np.int64)
        np.savez_compressed(cache_npz, waveforms=raw_windows, labels=labels)
        data_manifest = {
            "settings": cache_settings, "npz_sha256": sha256_file(cache_npz),
            "dataset": "ETHZ", "waveform_url": WAVEFORM_URL, "waveform_archive_bytes": WAVEFORM_BYTES,
            "tables": {name: {"url": MIRROR + source[0], "bytes": source[1], "sha256": source[2]} for name, source in TABLES.items()},
            "records": records, "sampling_rate_hz": 100, "component_order": "ZNE",
            "resampling": "200 Hz: scipy.signal.resample_poly(up=1, down=2), with 20 input samples of filter context where available; 100 Hz unchanged",
            "padding": False, "waveform_transfer_bytes": transfer_bytes, "waveform_requests": transfer_requests,
        }
        cache_json.write_text(json.dumps(data_manifest, indent=2), encoding="utf-8")

assert raw_windows.shape == (2 * SAMPLES_PER_CLASS, 3, 3000)
assert raw_windows.dtype == np.float32 and np.isfinite(raw_windows).all()
assert labels.dtype == np.int64 and np.array_equal(np.bincount(labels), [SAMPLES_PER_CLASS, SAMPLES_PER_CLASS])
assert len(data_manifest["records"]) == len(labels)
print(f"{SAMPLES_PER_CLASS} earthquake + {SAMPLES_PER_CLASS} noise windows | cache: {'reused' if cache_hit else 'created'}")
print(f"Waveform transfer when cached: {data_manifest['waveform_transfer_bytes'] / 1024**2:.2f} MiB")

## 4. Normalize and inspect

Demean each component and divide by its population standard deviation plus `1e-10`. Both encoders receive identical normalized windows.

In [ ]:
waveforms = torch.from_numpy(raw_windows.copy())
waveforms = waveforms - waveforms.mean(dim=-1, keepdim=True)
std = waveforms.std(dim=-1, correction=0, keepdim=True)
waveforms = waveforms / (torch.where(std == 0, torch.ones_like(std), std) + 1e-10)
assert torch.isfinite(waveforms).all()

colors = {"earthquake": "#C7614C", "noise": "#247C89"}
with plt.rc_context({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False}):
    fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True, constrained_layout=True)
    times = np.arange(3000) / 100
    for label, kind in enumerate(("earthquake", "noise")):
        index = int(np.flatnonzero(labels == label)[0])
        for component, trace, offset in zip("ZNE", waveforms[index].numpy(), (8, 0, -8)):
            axes[label].plot(times, trace + offset, linewidth=0.65, color=colors[kind], alpha=0.85)
        axes[label].set_yticks([8, 0, -8], labels=list("ZNE"))
        axes[label].set_title(kind.capitalize(), loc="left", fontweight="bold")
        axes[label].set_ylabel("Normalized input (offset)")
    axes[-1].set_xlabel("Time since window start [s]")
    axes[-1].set_xlim(0, 30)
    fig.savefig(OUTPUT_DIR / "example_windows.png", dpi=160)
    plt.show()

## 5. Extract pooled features

Load the Hugging Face model and create a seeded random model with the same configuration. Average each layer's output over time; run models sequentially to limit memory.

In [ ]:
pooled_features = {}
encoder_seconds = {}
model_config = None

for model_kind in ("pretrained", "random_init"):
    if model_kind == "pretrained":
        model = load_pretrained(release["repo_id"], revision=release["revision"], device=device, cache_dir=CACHE_DIR)
        model_config = model.config
    else:
        # Seed construction on CPU, then transfer; this also avoids CUDA RNG side effects.
        with torch.random.fork_rng(devices=[]):
            torch.random.default_generator.manual_seed(SEED)
            model = MultiDimWav2Vec2ForPreTraining(model_config)
        model = model.to(device).eval()
    layer_count = int(model.config.num_hidden_layers)
    selected_layer = HIDDEN_LAYER if HIDDEN_LAYER >= 0 else layer_count + 1 + HIDDEN_LAYER
    if not 0 <= selected_layer <= layer_count:
        raise ValueError(f"HIDDEN_LAYER must index 0–{layer_count}, or the corresponding negative index.")
    batches = collections.defaultdict(list)
    if device == "cuda":
        torch.cuda.synchronize()
    encoder_started = time.perf_counter()
    with torch.inference_mode():
        for start in range(0, len(waveforms), BATCH_SIZE):
            output = model.wav2vec2(waveforms[start:start + BATCH_SIZE].to(device), output_hidden_states=True)
            batches["conv_features"].append(output.extract_features.mean(dim=1).cpu().numpy())
            for layer, states in enumerate(output.hidden_states):
                batches[f"hidden_{layer}"].append(states.mean(dim=1).cpu().numpy())
    if device == "cuda":
        torch.cuda.synchronize()
    encoder_seconds[model_kind] = time.perf_counter() - encoder_started
    pooled_features[model_kind] = {name: np.concatenate(values) for name, values in batches.items()}
    for values in pooled_features[model_kind].values():
        assert values.shape[0] == len(labels) and values.dtype == np.float32 and np.isfinite(values).all()
    del output, model
    if device == "cuda":
        torch.cuda.empty_cache()
    print(f"{model_kind}: {encoder_seconds[model_kind]:.2f} s")

feature_key = f"hidden_{selected_layer}"
print("Selected representation:", feature_key, "| shape:", pooled_features["pretrained"][feature_key].shape)

## 6. Compare projections

t-SNE distorts distances. Each panel is fitted independently: **positions and axes are not comparable between panels**. Visual separation does not establish detection performance; that requires a downstream classifier and independent evaluation.

In [ ]:
perplexity = min(20.0, (len(labels) - 1) / 3)
projections = {}
with threadpool_limits(limits=2):
    for kind in ("random_init", "pretrained"):
        projections[kind] = TSNE(
            n_components=2, perplexity=perplexity, init="pca", learning_rate="auto",
            max_iter=750, random_state=SEED, n_jobs=2,
        ).fit_transform(pooled_features[kind][feature_key])
        assert projections[kind].shape == (len(labels), 2) and np.isfinite(projections[kind]).all()

with plt.rc_context({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False}):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), constrained_layout=True)
    for ax, kind, title in zip(axes, ("random_init", "pretrained"), ("Random initialization", "Pretrained on seismic waveforms")):
        for label, name, marker in ((0, "earthquake", "^"), (1, "noise", "o")):
            points = projections[kind][labels == label]
            ax.scatter(points[:, 0], points[:, 1], label=name.capitalize(), color=colors[name],
                       marker=marker, s=42, alpha=0.82, edgecolors="white", linewidths=0.45)
        ax.set_title(title, loc="left", fontweight="bold", pad=12)
        ax.set_xlabel("t-SNE 1")
        ax.set_ylabel("t-SNE 2")
        ax.tick_params(length=0)
        ax.grid(alpha=0.12)
    axes[0].legend(frameon=False, loc="best")
    fig.suptitle(f"SeisLM {MODEL_NAME} · layer {selected_layer} · ETHZ development windows", fontsize=13)
    fig.savefig(OUTPUT_DIR / "embedding_comparison.png", dpi=180)
    plt.show()

## 7. Explore the waveforms

Hover over a point in either projection to see its normalized Z/N/E input. Outlines link the same window across both plots.

The saved **interactive_embeddings.html** also works offline, without a running notebook kernel.

In [ ]:
import base64

import plotly.graph_objects as go
from IPython.display import HTML, display
from plotly.subplots import make_subplots


def make_interactive_embeddings(waves, coordinates, labels, records, *, sampling_rate_hz,
                                component_order, model_name, layer, revision):
    """Build a self-contained plot from the existing model inputs and projections."""
    waves = np.ascontiguousarray(waves, dtype="<f4")
    labels = np.asarray(labels)
    coordinates = {kind: np.asarray(coordinates[kind]) for kind in ("random_init", "pretrained")}
    if waves.ndim != 3 or waves.shape[1] != 3 or not waves.shape[0] or not waves.shape[2]:
        raise ValueError("Expected nonempty [window, 3, sample] waveforms.")
    if labels.shape != (len(waves),) or not set(labels.tolist()) <= {0, 1} or len(records) != len(waves):
        raise ValueError("Labels and records must match the waveform rows.")
    if not np.isfinite(waves).all() or any(
        xy.shape != (len(waves), 2) or not np.isfinite(xy).all() for xy in coordinates.values()
    ):
        raise ValueError("Waveforms and matching 2-D coordinates must be finite.")
    if sampling_rate_hz <= 0 or not np.isfinite(sampling_rate_hz) or component_order != "ZNE":
        raise ValueError("Use a positive sampling rate and ZNE component order.")

    offsets = [8, 0, -8]
    times = np.arange(waves.shape[-1]) / sampling_rate_hz
    fig = make_subplots(
        rows=2, cols=2, specs=[[{}, {}], [{"colspan": 2}, None]],
        row_heights=[0.6, 0.4], horizontal_spacing=0.12, vertical_spacing=0.18,
        subplot_titles=["Random initialization", "Seismic pretraining", "Normalized model input"],
    )
    for column, kind in enumerate(("random_init", "pretrained"), start=1):
        for label, name, symbol, color in (
            (0, "Earthquake", "triangle-up", "#C7614C"), (1, "Noise", "circle", "#247C89")
        ):
            ids = np.flatnonzero(labels == label)
            xy = coordinates[kind][ids]
            fig.add_trace(go.Scatter(
                x=xy[:, 0], y=xy[:, 1], customdata=ids.tolist(),
                meta={"role": "embedding", "projection": kind}, mode="markers",
                name=name, legendgroup=str(label), showlegend=column == 1,
                marker={"color": color, "symbol": symbol, "size": 11, "opacity": 0.86,
                        "line": {"width": 1, "color": "white"}},
                hovertemplate=name + " · window %{customdata}<extra></extra>",
            ), row=1, col=column)
    highlight_indices = []
    for column, kind in enumerate(("random_init", "pretrained"), start=1):
        highlight_indices.append(len(fig.data))
        fig.add_trace(go.Scatter(
            x=[float(coordinates[kind][0, 0])], y=[float(coordinates[kind][0, 1])],
            mode="markers", showlegend=False, hoverinfo="skip", meta={"role": "highlight"},
            marker={"size": 20, "symbol": "circle-open", "color": "#263445", "line": {"width": 2}},
        ), row=1, col=column)
    waveform_indices = []
    for component, color, offset, values in zip(
        component_order, ("#247C89", "#C7614C", "#8274A3"), offsets, waves[0]
    ):
        waveform_indices.append(len(fig.data))
        fig.add_trace(go.Scatter(
            x=times.tolist(), y=(values.astype(np.float64) + offset).tolist(), mode="lines",
            name=component, showlegend=False, meta={"role": "waveform"},
            line={"color": color, "width": 1}, hoverinfo="skip",
        ), row=2, col=1)
    fig.update_layout(
        title={"text": f"SeisLM {model_name} · layer {layer} · waveform explorer", "x": 0.035,
               "font": {"size": 21}},
        template="plotly_white", height=800, margin={"l": 72, "r": 35, "t": 130, "b": 65},
        font={"family": "Arial, sans-serif", "color": "#263445", "size": 12},
        paper_bgcolor="#FBFCFE", plot_bgcolor="#FBFCFE",
        legend={"orientation": "h", "x": 0, "y": 1.08}, hovermode="closest",
    )
    for column in (1, 2):
        fig.update_xaxes(title_text="t-SNE 1", zeroline=False, gridcolor="#E9EDF2", row=1, col=column)
        fig.update_yaxes(title_text="t-SNE 2", zeroline=False, gridcolor="#E9EDF2", row=1, col=column)
    fig.update_xaxes(title_text="Time since window start [s]", range=[0, len(times) / sampling_rate_hz],
                     gridcolor="#E9EDF2", zeroline=False, row=2, col=1)
    limit = float(np.max(np.abs(waves[0]))) + 9
    fig.update_yaxes(title_text="Component · normalized amplitude (offset)",
                     tickvals=offsets, ticktext=list(component_order), range=[-limit, limit],
                     gridcolor="#E9EDF2", zeroline=False, row=2, col=1)

    # Lossless float32 storage keeps every sample without large decimal JSON arrays.
    payload = {
        "waveforms_f32_le": base64.b64encode(waves.tobytes()).decode("ascii"),
        "shape": list(waves.shape), "labels": labels.tolist(), "records": records,
        "projections": {kind: xy.tolist() for kind, xy in coordinates.items()},
        "sampling_rate_hz": float(sampling_rate_hz), "component_order": component_order,
        "model": model_name, "layer": int(layer), "revision": revision,
    }
    encoded = json.dumps(payload, separators=(",", ":"), ensure_ascii=True)
    encoded = encoded.replace("<", "\\u003c").replace(">", "\\u003e").replace("&", "\\u0026")
    script = r"""
(() => {
  const gd = document.getElementById('{plot_id}');
  const embedded = __DATA__;
  const waveformIndices = __WAVEFORM_INDICES__;
  const highlightIndices = __HIGHLIGHT_INDICES__;
  const offsets = [8, 0, -8];
  const bytes = Uint8Array.from(atob(embedded.waveforms_f32_le), c => c.charCodeAt(0));
  const view = new DataView(bytes.buffer);
  const samples = new Float32Array(bytes.length / 4);
  for (let i = 0; i < samples.length; i++) samples[i] = view.getFloat32(4 * i, true);
  const escapeText = s => String(s).replace(/[&<>"']/g, c => ({
    '&': '&amp;', '<': '&lt;', '>': '&gt;', '"': '&quot;', "'": '&#39;'
  })[c]);
  let selected = -1, pending = null, drawing = false;

  async function selectWindow(id) {
    if (!Number.isInteger(id) || id < 0 || id >= embedded.shape[0]) return;
    pending = id;
    if (drawing) return;
    drawing = true;
    try {
      while (pending !== null) {
        const next = pending;
        pending = null;
        if (next === selected) continue;
        const n = embedded.shape[2];
        const wave = offsets.map((_, c) => samples.subarray((3 * next + c) * n, (3 * next + c + 1) * n));
        const record = embedded.records[next];
        const label = embedded.labels[next] === 0 ? 'Earthquake' : 'Noise';
        const recording = record.trace_name_original || record.trace_name || 'recording';
        let peak = 0;
        for (const channel of wave) for (const value of channel) peak = Math.max(peak, Math.abs(value));
        await Plotly.restyle(gd, {y: wave.map((channel, c) => Array.from(channel, y => y + offsets[c]))}, waveformIndices);
        const xy = ['random_init', 'pretrained'].map(kind => embedded.projections[kind][next]);
        await Plotly.restyle(gd, {x: xy.map(p => [p[0]]), y: xy.map(p => [p[1]])}, highlightIndices);
        await Plotly.relayout(gd, {
          'yaxis3.range': [-9 - peak, 9 + peak],
          'annotations[2].text': 'Normalized model input · ' + escapeText(label) +
            ' · window ' + next + '<br><span style="font-size:11px;color:#65758B">' + escapeText(recording) + '</span>'
        });
        selected = next;
        gd.dataset.selectedId = String(next);
        gd.dataset.selectedTrace = String(recording);
        gd.dataset.selectedClass = label;
      }
    } finally {
      drawing = false;
    }
  }

  gd.on('plotly_hover', event => {
    const point = event.points && event.points[0];
    if (point?.data.meta?.role === 'embedding') selectWindow(Number(point.customdata));
  });
  gd.dataset.seislmInteractive = 'true';
  gd.dataset.modelRevision = embedded.revision;
  gd.dataset.selectedLayer = String(embedded.layer);
  selectWindow(0).then(() => { gd.dataset.ready = 'true'; });
})();
"""
    script = (script.replace("__DATA__", encoded)
              .replace("__WAVEFORM_INDICES__", json.dumps(waveform_indices))
              .replace("__HIGHLIGHT_INDICES__", json.dumps(highlight_indices)))
    fragment = fig.to_html(
        full_html=False, include_plotlyjs=True, include_mathjax=False,
        div_id="seislm-" + uuid.uuid4().hex, post_script=script,
        config={"responsive": True, "displaylogo": False, "scrollZoom": False},
    )
    # Keep each output's callbacks bound to the Plotly instance that created it.
    # Loading another output can replace window.Plotly before newPlot resolves.
    prefix, marker, renderer = fragment.rpartition("<script>")
    if not marker or "Plotly.newPlot" not in renderer:
        raise RuntimeError("Unexpected Plotly HTML format.")
    return (prefix + marker + "(function(Plotly) {" +
            renderer.replace("</script>", "})(window.Plotly);</script>", 1))

In [ ]:
interactive_fragment = make_interactive_embeddings(
    waveforms.numpy(), projections, labels, data_manifest["records"],
    sampling_rate_hz=data_manifest["sampling_rate_hz"],
    component_order=data_manifest["component_order"],
    model_name=MODEL_NAME, layer=selected_layer, revision=release["revision"],
)
interactive_path = OUTPUT_DIR / "interactive_embeddings.html"
interactive_path.write_text(
    '<!doctype html><html><head><meta charset="utf-8">'
    '<meta name="viewport" content="width=device-width,initial-scale=1">'
    '<title>SeisLM waveform explorer</title>'
    '<style>body{margin:0;background:#FBFCFE;font-family:Arial,sans-serif;}'
    'p{margin:24px 32px 0;color:#65758B;font-size:14px;}</style></head><body>'
    '<p>Hover over a point to inspect its normalized Z/N/E waveform. '
    'Outlines link the same window across both projections.</p>'
    + interactive_fragment + '</body></html>', encoding="utf-8",
)
display(HTML(interactive_fragment))

## 8. Save and reuse

The output folder contains plots, normalized waveforms, pooled features, coordinates, labels, and data provenance. Download them through Colab's **Files** pane. Reload arrays with `np.load(..., allow_pickle=False)`.

For your data, replace the ETHZ loader with float32 **[window, 3, sample]** arrays at **100 Hz in ZNE order** and trustworthy labels. Keep preprocessing consistent; split by event before training a classifier.

In [ ]:
numeric_outputs = {f"{kind}__{name}": values for kind, layers in pooled_features.items() for name, values in layers.items()}
numeric_outputs.update({f"{kind}__tsne": values for kind, values in projections.items()})
numeric_outputs["labels"] = labels
numeric_outputs["waveforms"] = waveforms.numpy()
np.savez_compressed(OUTPUT_DIR / "representations.npz", **numeric_outputs)
(OUTPUT_DIR / "data-provenance.json").write_text(json.dumps(data_manifest, indent=2), encoding="utf-8")
summary = {
    "notebook": "feature_visualization", "model": {"name": MODEL_NAME, **release},
    "versions": versions, "device": device, "seed": SEED,
    "input_shape": list(waveforms.shape), "pooled_feature_shape": list(pooled_features["pretrained"][feature_key].shape),
    "selected_layer": selected_layer, "labels": {"0": "earthquake", "1": "noise"},
    "class_counts": np.bincount(labels).tolist(), "dataset": "ETHZ", "split": "dev",
    "sampling_rate_hz": 100, "component_order": "ZNE", "padding": False,
    "normalization": "per-component demean and population std + 1e-10; zero std replaced by 1",
    "data_cache_hit": cache_hit, "data_cache_sha256": data_manifest["npz_sha256"],
    "waveform_transfer_bytes": 0 if cache_hit else data_manifest["waveform_transfer_bytes"],
    "tsne": {"perplexity": perplexity, "max_iter": 750, "seed": SEED, "independent_projections": True},
    "interactive": {"waveform_encoding": "lossless float32 little-endian base64",
                    "samples_per_component": int(waveforms.shape[-1]), "offline_html": True},
    "encoder_seconds": encoder_seconds, "elapsed_seconds": time.perf_counter() - started,
    "artifacts": ["example_windows.png", "embedding_comparison.png", "interactive_embeddings.html", "representations.npz", "data-provenance.json"],
}
(OUTPUT_DIR / "run-summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
with np.load(OUTPUT_DIR / "representations.npz", allow_pickle=False) as saved:
    for key, values in numeric_outputs.items():
        np.testing.assert_array_equal(saved[key], values)
for name in summary["artifacts"]:
    assert (OUTPUT_DIR / name).is_file()
print("Saved:", ", ".join(summary["artifacts"]))
print("Output folder:", OUTPUT_DIR)

When using these data, cite [ETHZ's contributing networks](https://seisbench.readthedocs.io/en/main/pages/data/benchmark_datasets.html#ethz), [SeisBench](https://doi.org/10.1785/0220210324), and the [picking benchmark](https://github.com/seisbench/pick-benchmark). Cite [SeisLM](https://arxiv.org/abs/2410.15765) when using the model.